<a href="https://colab.research.google.com/github/Cluemore/TestriQA-Internship-/blob/main/Zero_shot_text_summarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch transformers rouge-score sentence-transformers datasets huggingface-hub

from transformers import pipeline
from rouge_score import rouge_scorer
import pandas as pd

  Preparing metadata (setup.py) ... done


In [ ]:
# Load summarization model
summarizer = pipeline("text-generation", model="facebook/bart-large-cnn")

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/316 [00:00<?, ?it/s]

BartForCausalLM LOAD REPORT from: facebook/bart-large-cnn
Key                                                       | Status     |  | 
----------------------------------------------------------+------------+--+-
model.encoder.layers.{0...11}.self_attn.v_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.q_proj.bias       | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.bias   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc1.weight                  | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.k_proj.weight     | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn.out_proj.weight   | UNEXPECTED |  | 
model.encoder.layers.{0...11}.fc2.bias                    | UNEXPECTED |  | 
model.encoder.layers.{0...11}.self_attn_layer_norm.weight | UNEXPECTED |  | 
model.encoder.laye

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
articles = []
print("Please paste 5 long news articles one by one. After pasting each article, type 'END' on a new line.")

for i in range(1, 6):
    print(f"\nPaste article #{i}:")
    lines = []
    while True:
        line = input()
        if line.strip() == "END":
            break
        lines.append(line)
    article_text = "\n".join(lines)
    articles.append(article_text)

Please paste 5 long news articles one by one. After pasting each article, type 'END' on a new line.

Paste article #1:
Nvidia will launch a new artificial intelligence chipset for China at a significantly lower price than its recently restricted H20 model and plans to start mass production as early as June, sources familiar with the matter said.  The GPU or graphics processing unit will be part of Nvidia’s latest generation Blackwell-architecture AI processors and is expected to be priced between $6,500 and $8,000, well below the $10,000-$12,000 the H20 sold for, according to two of the sources.  The lower price reflects its weaker specifications and simpler manufacturing requirements.  It will be based on Nvidia’s RTX Pro 6000D, a server-class graphics processor and will use conventional GDDR7 memory instead of more advanced high bandwidth memory, the two sources said. The new chip’s price, specifications and production timing have not previously been reported.  The three sources Reut

In [ ]:
# Define prompt templates for zero-shot summarization variations
prompt_templates = [
    {"prompt": "Summarize the given news article in 3–4 complete and well-structured sentences suitable for a college-level analysis. Ensure the summary includes the main idea, key developments or events, and any significant outcomes or implications. Use formal language and avoid vague phrasing. The summary should reflect a clear understanding of the topic and provide enough context for readers unfamiliar with the article.", "max_length": 100, "min_length": 60},
    {"prompt": "Briefly summarize the following news article in 1–2 concise sentences. Ensure the summary captures the main point or event and includes the most important facts or developments. Additionally, provide 2–3 key bullet points highlighting critical details such as names, dates, statistics, decisions, or consequences mentioned in the article. The summary should be easy to understand for someone unfamiliar with the topic, and the bullet points should emphasize the most relevant information without repeating the summary.", "max_length": 80, "min_length": 40},
    {"prompt": "Write a comprehensive and detailed summary of the article, consisting of 5 to 7 well-structured sentences. Focus on highlighting the key facts, main events, and important figures or organizations involved. Be sure to explain the context of the news clearly, including any background information necessary to understand the topic. Provide a clear and informative explanation of why the news is important, what its potential impact might be, and how it relates to current trends or issues. Avoid vague statements and aim to convey the core message of the article in a concise but thorough manner. Make the summary easy to understand for someone who has no prior knowledge of the topic.", "max_length": 150, "min_length": 100},
]

results = []

for idx, text in enumerate(articles, 1):
    print(f"\nProcessing Article #{idx}...")
    for pt in prompt_templates:
        input_text = pt['prompt'] + "\n\n" + text
        summary = summarizer(input_text, max_length=pt['max_length'], min_length=pt['min_length'], do_sample=False)[0]['generated_text']
        scores = scorer.score(text, summary)

        results.append({
            "article_num": idx,
            "prompt": pt['prompt'],
            "max_length": pt['max_length'],
            "min_length": pt['min_length'],
            "summary": summary,
            "rouge1_f1": scores['rouge1'].fmeasure,
            "rouge2_f1": scores['rouge2'].fmeasure,
            "rougeL_f1": scores['rougeL'].fmeasure
        })

        print(f"Prompt: {pt['prompt']}")
        print(f"Summary:\n{summary}\n")
        print(f"ROUGE-1 F1: {scores['rouge1'].fmeasure:.4f}, ROUGE-2 F1: {scores['rouge2'].fmeasure:.4f}, ROUGE-L F1: {scores['rougeL'].fmeasure:.4f}")
        print("-" * 60)


Processing Article #1...
Prompt: Summarize the given news article in 3–4 complete and well-structured sentences suitable for a college-level analysis. Ensure the summary includes the main idea, key developments or events, and any significant outcomes or implications. Use formal language and avoid vague phrasing. The summary should reflect a clear understanding of the topic and provide enough context for readers unfamiliar with the article.
Summary:
Summarize the given news article in 3–4 complete and well-structured sentences suitable for a college-level analysis. Ensure the summary includes the main idea, key developments or events, and any significant outcomes or implications. Use formal language and avoid vague phrasing. The summary should reflect a clear understanding of the topic and provide enough context for readers unfamiliar with the article.

Nvidia will launch a new artificial intelligence chipset for China at a significantly lower price than its recently restricted H20 mod

In [ ]:
df = pd.DataFrame(results)
df.to_csv("zero_shot_summarization_results.csv", index=False)
print("\nSaved all summaries and ROUGE scores to 'zero_shot_summarization_results.csv'")



Saved all summaries and ROUGE scores to 'zero_shot_summarization_results.csv'
